In [1]:
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="retrieval_demo",
    embedding_function=embedder
)

docs = [
    "Python is commonly used for automation, data analysis, and machine learning.",
    "A SQL database stores structured information using tables, rows, and columns.",
    "Vector databases retrieve information using embeddings and similarity search.",
    "REST APIs allow software systems to exchange data using HTTP operations.",
    "Machine learning models learn patterns from examples and use them to make predictions."
]

ids = [f"doc_{i}" for i in range(len(docs))]
metas = [
    {"source": "python.txt", "topic": "programming"},
    {"source": "sql.txt", "topic": "database"},
    {"source": "vectors.txt", "topic": "vector_search"},
    {"source": "api.txt", "topic": "web"},
    {"source": "ml.txt", "topic": "machine_learning"}
]

collection.add(ids=ids, documents=docs, metadatas=metas)
print("Documents:", collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents: 5


In [3]:
user_query = "How are embeddings used to search information?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for rank, (doc, meta, distance) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1
):
    print(f"Rank {rank}")
    print("Distance:", round(distance, 4))
    print("Topic:", meta["topic"])
    print("Text:", doc)
    print()


Rank 1
Distance: 0.3369
Topic: vector_search
Text: Vector databases retrieve information using embeddings and similarity search.

Rank 2
Distance: 0.685
Topic: machine_learning
Text: Machine learning models learn patterns from examples and use them to make predictions.

Rank 3
Distance: 0.7191
Topic: database
Text: A SQL database stores structured information using tables, rows, and columns.



In [4]:
def distance_to_similarity(distance):
    return 1 - distance

for doc, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(
        f"Similarity: {distance_to_similarity(distance):.4f} | {doc}"
    )


Similarity: 0.6631 | Vector databases retrieve information using embeddings and similarity search.
Similarity: 0.3150 | Machine learning models learn patterns from examples and use them to make predictions.
Similarity: 0.2809 | A SQL database stores structured information using tables, rows, and columns.


In [5]:
def retrieve_with_threshold(query, n_results=5, threshold=0.30):
    result = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    matches = []
    for doc, meta, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        similarity = 1 - distance
        if similarity >= threshold:
            matches.append({
                "document": doc,
                "metadata": meta,
                "similarity": similarity
            })
    return matches

matches = retrieve_with_threshold(
    "What is a vector database?",
    threshold=0.30
)

for match in matches:
    print(round(match["similarity"], 4), match["document"])


0.6561 Vector databases retrieve information using embeddings and similarity search.
0.5493 A SQL database stores structured information using tables, rows, and columns.
0.3096 Machine learning models learn patterns from examples and use them to make predictions.


In [6]:
unrelated = retrieve_with_threshold(
    "How do I repair a bicycle tire?",
    threshold=0.30
)

print("Relevant matches:", len(unrelated))
for item in unrelated:
    print(item)


Relevant matches: 0


In [7]:
filtered = collection.query(
    query_texts=["How do systems communicate over HTTP?"],
    n_results=2,
    where={"topic": "web"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    filtered["documents"][0],
    filtered["metadatas"][0],
    filtered["distances"][0]
):
    print(meta["source"], round(1 - distance, 4), doc)


api.txt 0.561 REST APIs allow software systems to exchange data using HTTP operations.


In [8]:
def retrieve_context(
    query,
    n_results=4,
    similarity_threshold=0.25,
    topic_filter=None
):
    kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }

    if topic_filter:
        kwargs["where"] = {"topic": topic_filter}

    result = collection.query(**kwargs)

    context = []
    for doc, meta, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):
        similarity = 1 - distance
        if similarity >= similarity_threshold:
            context.append({
                "text": doc,
                "source": meta["source"],
                "similarity": similarity
            })

    return context

context = retrieve_context(
    "What technology stores embeddings for similarity search?"
)

for item in context:
    print(item)


{'text': 'Vector databases retrieve information using embeddings and similarity search.', 'source': 'vectors.txt', 'similarity': 0.6897832155227661}
{'text': 'Machine learning models learn patterns from examples and use them to make predictions.', 'source': 'ml.txt', 'similarity': 0.28456878662109375}
{'text': 'A SQL database stores structured information using tables, rows, and columns.', 'source': 'sql.txt', 'similarity': 0.28109312057495117}


In [9]:
def format_context(query, retrieved):
    if not retrieved:
        return (
            f"Question: {query}\n\n"
            "No relevant information was found in the knowledge base."
        )

    context_text = "\n\n".join(
        f"[Source: {item['source']}]\n{item['text']}"
        for item in retrieved
    )

    return f"""Use only the following retrieved context to answer the question.

Context:
{context_text}

Question: {query}

Answer:"""

final_query = "What is a vector database?"
retrieved = retrieve_context(final_query)
prompt = format_context(final_query, retrieved)

print(prompt)


Use only the following retrieved context to answer the question.

Context:
[Source: vectors.txt]
Vector databases retrieve information using embeddings and similarity search.

[Source: sql.txt]
A SQL database stores structured information using tables, rows, and columns.

[Source: ml.txt]
Machine learning models learn patterns from examples and use them to make predictions.

Question: What is a vector database?

Answer:


In [10]:
questions = [
    "What are embeddings useful for?",
    "How does a database organize tabular information?",
    "What is used to expose software functionality through HTTP?"
]

for q in questions:
    matches = retrieve_context(q)
    print("\nQUESTION:", q)
    for item in matches[:2]:
        print(
            f"  {item['similarity']:.4f} -> "
            f"{item['text']}"
        )



QUESTION: What are embeddings useful for?
  0.4759 -> Vector databases retrieve information using embeddings and similarity search.

QUESTION: How does a database organize tabular information?
  0.6673 -> A SQL database stores structured information using tables, rows, and columns.
  0.3226 -> Vector databases retrieve information using embeddings and similarity search.

QUESTION: What is used to expose software functionality through HTTP?
  0.7087 -> REST APIs allow software systems to exchange data using HTTP operations.
